# Poglavlje 7: Izrada chat aplikacija
## Microsoft Foundry Models API Brzi početak

Ovaj bilježnik prilagođen je iz [Azure OpenAI Primjeri repozitorij](https://github.com/Azure/azure-openai-samples?WT.mc_id=academic-105485-koreyst) koji uključuje bilježnice koje pristupaju [Azure OpenAI](notebook-azure-openai.ipynb) uslugama.

> **Napomena:** GitHub Models se ukida krajem srpnja 2026. Ovaj bilježnik sada cilja na [Microsoft Foundry Models](https://ai.azure.com/catalog/models?WT.mc_id=academic-105485-koreyst), koji nudi isti katalog modela besplatnih za isprobavanje i iskustvo Azure AI Inference SDK-a.


# Pregled  
"Veliki jezični modeli su funkcije koje mapiraju tekst na tekst. Dajući ulazni niz teksta, veliki jezični model pokušava predvidjeti tekst koji će uslijediti"(1). Ovaj "quickstart" bilježnica će korisnike upoznati s konceptima visokog nivoa LLM-a, osnovnim paketima potrebnim za početak s AML-om, nježnim uvodom u dizajn upita te nekoliko kratkih primjera različitih slučajeva korištenja. 


## Sadržaj  

[Pregled](#overview)  
[Kako koristiti OpenAI uslugu](#how-to-use-openai-service)  
[1. Kreiranje vaše OpenAI usluge](#1.-creating-your-openai-service)  
[2. Instalacija](#2.-installation)    
[3. Autentifikacijski podaci](#3.-credentials)  

[Primjeri uporabe](#use-cases)    
[1. Sažetak teksta](#1.-summarize-text)  
[2. Klasificiraj tekst](#2.-classify-text)  
[3. Generiraj nova imena proizvoda](#3.-generate-new-product-names)  
[4. Fino podešavanje klasifikatora](#4.fine-tune-a-classifier)  

[Reference](#references)


### Izradite svoj prvi upit  
Ova kratka vježba pružit će osnovni uvod u slanje upita modelu u Microsoft Foundry Models za jednostavan zadatak "sažimanje".


**Koraci**:  
1. Instalirajte `azure-ai-inference` biblioteku u vašem Python okruženju, ako to već niste napravili.  
2. Učitajte standardne pomoćne biblioteke i postavite svoje vjerodajnice za Microsoft Foundry Models.  
3. Odaberite model za svoj zadatak  
4. Kreirajte jednostavan upit za model  
5. Pošaljite svoj zahtjev model API-ju!


### 1. Instalirajte `azure-ai-inference`


In [ ]:
%pip install azure-ai-inference

### 2. Uvoz pomoćnih biblioteka i instanciranje vjerodajnica


In [ ]:
import os
from azure.ai.inference import ChatCompletionsClient
from azure.ai.inference.models import SystemMessage, UserMessage
from azure.core.credentials import AzureKeyCredential

# Get these from your Microsoft Foundry project's "Overview" page
token = os.environ["AZURE_INFERENCE_CREDENTIAL"]
endpoint = os.environ["AZURE_INFERENCE_ENDPOINT"]

client = ChatCompletionsClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(token),
)


### 3. Pronalaženje pravog modela  
Modeli poput GPT-4o i GPT-4o mini mogu razumjeti i generirati prirodni jezik, a dostupni su u katalogu Microsoft Foundry Models zajedno s modelima iz Meta, Mistral, Cohere i Microsoft.


In [ ]:
# Select a general purpose chat model
model_name = "gpt-4o-mini"


## 4. Dizajn upita  

"Čarolija velikih jezičnih modela je u tome što se treniranjem da minimiziraju ovu pogrešku u predviđanju preko golemih količina teksta, modeli nauče koncepte korisne za ova predviđanja. Na primjer, nauče koncepte poput"(1):

* kako se piše
* kako funkcionira gramatika
* kako preformulirati
* kako odgovarati na pitanja
* kako voditi razgovor
* kako pisati na mnogim jezicima
* kako kodirati
* itd.

#### Kako kontrolirati veliki jezični model  
"Od svih ulaza za veliki jezični model, daleko najvažniji je tekstualni upit(1).

Veliki jezični modeli se mogu potaknuti da generiraju izlaz na nekoliko načina:

Instrukcija: Reci modelu što želiš
Dovršavanje: Potakni model da dovrši početak onoga što želiš
Demonstracija: Pokaži modelu što želiš, s:
Nekoliko primjera u upitu
Stotinama ili tisućama primjera u skupu podataka za fino podešavanje"



#### Postoje tri osnovna pravila za izradu upita:

**Pokaži i reci**. Jasno pokaži što želiš bilo kroz instrukcije, primjere ili kombinaciju oboje. Ako želiš da model rangira popis stavki po abecedi ili da klasificira odlomak prema sentimentu, pokaži mu da je to ono što želiš.

**Osiguraj kvalitetne podatke**. Ako pokušavaš izgraditi klasifikator ili natjerati model da slijedi obrazac, pobrini se da ima dovoljno primjera. Obavezno provjeri svoje primjere — model je obično dovoljno pametan da uoči osnovne pravopisne greške i da ti odgovor, ali isto tako može pretpostaviti da je to namjerno i to može utjecati na odgovor.

**Provjeri svoje postavke.** Postavke temperature i top_p kontroliraju koliko je model determinističan pri generiranju odgovora. Ako tražiš odgovor gdje postoji samo jedan točan odgovor, trebalo bi ih postaviti niže. Ako tražiš raznovrsnije odgovore, možda ih želiš postaviti više. Najčešća pogreška koju ljudi rade s tim postavkama je pretpostavka da su to kontrole "pametnosti" ili "kreativnosti".


Izvor: https://learn.microsoft.com/azure/ai-services/openai/overview


### 5. Pošalji!


In [ ]:
# Create your first prompt
text_prompt = "Should oxford commas always be used?"

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

### Ponovite isti poziv, kako se rezultati uspoređuju?


In [ ]:

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":text_prompt},])

response.choices[0].message.content

## Sažmi tekst  
#### Izazov  
Sažmi tekst dodavanjem 'tl;dr:' na kraj odlomka teksta. Primijetite kako model razumije kako izvršiti niz zadataka bez dodatnih uputa. Možete eksperimentirati s opisnijim uputama od tl;dr kako biste izmijenili ponašanje modela i prilagodili sažimanje koje primate(3).  

Nedavni radovi pokazali su značajne pomake u mnogim NLP zadacima i mjerilima kroz prethodno učenje na velikom korpusu teksta, praćeno dodatnim prilagođavanjem na specifičnom zadatku. Iako je arhitektura uglavnom nezavisna od zadatka, ova metoda još uvijek zahtijeva skupove podataka za fino podešavanje specifične za zadatak, od tisuća do desetaka tisuća primjera. Suprotno tome, ljudi općenito mogu obaviti novi jezični zadatak s nekoliko primjera ili jednostavnim uputama – nešto što trenutni NLP sustavi još uvijek uglavnom ne mogu. Ovdje prikazujemo da povećavanje veličine jezičnih modela značajno poboljšava izvedbu u zadatcima s nekoliko primjera, ponekad čak dostižući konkurentnost s dosadašnjim najboljim metodama finog podešavanja.  



Tl;dr


# Vježbe za nekoliko slučajeva upotrebe  
1. Sažmi tekst  
2. Klasificiraj tekst  
3. Generiraj nova imena proizvoda


In [ ]:
prompt = "Recent work has demonstrated substantial gains on many NLP tasks and benchmarks by pre-training on a large corpus of text followed by fine-tuning on a specific task. While typically task-agnostic in architecture, this method still requires task-specific fine-tuning datasets of thousands or tens of thousands of examples. By contrast, humans can generally perform a new language task from only a few examples or from simple instructions - something that current NLP systems still largely struggle to do. Here we show that scaling up language models greatly improves task-agnostic, few-shot performance, sometimes even reaching competitiveness with prior state-of-the-art fine-tuning approaches.\n\nTl;dr"


In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Klasificiraj tekst  
#### Izazov  
Klasificirajte stavke u kategorije dane u vrijeme izvođenja. U sljedećem primjeru, pružamo i kategorije i tekst za klasifikaciju u upitu(*playground_reference). 

Upit kupca: Zdravo, jedna tipka na tipkovnici mog laptopa nedavno se pokvarila i trebat će mi zamjena:

Klasificirana kategorija:


In [ ]:
prompt = "Classify the following inquiry into one of the following: categories: [Pricing, Hardware Support, Software Support]\n\ninquiry: Hello, one of the keys on my laptop keyboard broke recently and I'll need a replacement:\n\nClassified category:"
print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt},])

response.choices[0].message.content

## Generirajte Nove Nazive Proizvoda
#### Izazov
Kreirajte nazive proizvoda iz primjera riječi. Ovdje u upitu uključujemo informacije o proizvodu za koji ćemo generirati nazive. Također pružamo sličan primjer kako bismo pokazali obrazac koji želimo dobiti. Postavili smo i visoku vrijednost temperature kako bismo povećali slučajnost i dobili inovativnije odgovore.

Opis proizvoda: Kućni aparat za pripremu milkshakea
Početne riječi: brz, zdrav, kompaktan.
Nazivi proizvoda: HomeShaker, Fit Shaker, QuickShake, Shake Maker

Opis proizvoda: Par cipela koji može odgovarati na bilo koju veličinu stopala.
Početne riječi: prilagodljiv, odgovara, omni-fit.


In [ ]:
prompt = "Product description: A home milkshake maker\nSeed words: fast, healthy, compact.\nProduct names: HomeShaker, Fit Shaker, QuickShake, Shake Maker\n\nProduct description: A pair of shoes that can fit any foot size.\nSeed words: adaptable, fit, omni-fit."

print(prompt)

In [ ]:
#Setting a few additional, typical parameters during API Call

response = client.complete(
  model=model_name,
  messages = [{"role":"system", "content":"You are a helpful assistant."},
               {"role":"user","content":prompt}])

response.choices[0].message.content

# Reference  
- [Openai Cookbook](https://github.com/openai/openai-cookbook?WT.mc_id=academic-105485-koreyst)  
- [Microsoft Foundry portal](https://ai.azure.com?WT.mc_id=academic-105485-koreyst)  
- [Najbolje prakse za fino podešavanje GPT modela za klasifikaciju teksta](https://platform.openai.com/docs/guides/fine-tuning?WT.mc_id=academic-105485-koreyst)


# Za dodatnu pomoć  
[OpenAI komercijalizacijski tim](AzureOpenAITeam@microsoft.com) 


# Suradnici
* [Chew-Yean Yam](https://www.linkedin.com/in/cyyam/)


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Napomena**:
Ovaj dokument je preveden korištenjem AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako težimo točnosti, imajte na umu da automatski prijevodi mogu sadržavati greške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za važne informacije preporuča se profesionalni ljudski prijevod. Nismo odgovorni za bilo kakva nesporazumevanja ili pogrešne interpretacije koje proizlaze iz korištenja ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
